# importy

In [36]:
import requests
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

True

In [37]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [54]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [55]:
model = "anthropic/claude-sonnet-4.6"

In [58]:
class RiskAssessment(BaseModel):
    recommendation: str = Field(description="Wartość z listy: safe, risky, dangerous")
    justification: str = Field(description="Krótkie uzasadnienie rekomendacji")

In [59]:
system_message = """
Dokonujesz oceny ryzyka dla wędrówki górskiej dla wskazanego punktu w Tatrach.
Ocenę wykonujesz na podstawie 24-godzinnej prognozy (prognozowana temperatura co 3 godziny).
W odpowiedzi wskazujesz swoją ocenę poziomu ryzyka oraz krótkie uzasadnienie.
"""

In [60]:
with open("../data/json/20260321_162717.json", encoding='utf-8') as f:
    data = json.load(f)
forecast_dict = dict(list(data[0]['temperatures'].items())[:8])

In [65]:
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": json.dumps(forecast_dict)}
]

In [66]:
response = client.responses.parse(
    model=model,
    input=messages,
    text_format=RiskAssessment
)

In [67]:
response.output_parsed.model_dump()

{'recommendation': 'risky',
 'justification': 'Temperatury w nocy spadają do około -7°C, co stwarza ryzyko hipotermii i oblodzenia szlaków. Rano i w południe temperatura oscyluje wokół 0°C, co oznacza warunki lodowate i możliwość topnienia śniegu. Wyprawa wymaga odpowiedniego ekwipunku (raki, czekan), dobrego przygotowania i ostrożności, szczególnie we wczesnych godzinach.'}